# Scoring and audit, part 1, stage 1

Scores the six stage-main caches with the gate-E scorer and reads what needs no human
(DESIGN_ADDENDUM O, D270 to D273). Script: `scripts/scoring_audit.py`.

**Runtime: CPU.** No API call, no GPU, no cost. Run the cells in order. Every cell prints ids
and counts only. Files that hold answer text are named `TEXT_*.csv` and stay in Drive.

People: one runner. The runner reads the outputs, so the runner is never an HT-4 checker.
Nobody who may become an HT-4 checker opens this notebook's outputs or Drive `scored/`
(D168).

Before Cell 0: `scripts/scoring_audit.py` and this notebook are committed on `main` of
`banglaindoorvqa-main` (Cell 0 line C06 fails otherwise). Commit the notebook with its outputs
cleared.

Send back in the chat: the printout of Cells 0 to 7, and `scored/main/reading/TEXT_sample10.csv`
for the 10-item check.

## Cell P: settings
Set `REPO_SLUG` to the owner of the repo `banglaindoorvqa-main` (not the dataset repo
`banglaindoorvqa`). Nothing else changes.

In [ ]:
# Cell P: settings
DRIVE_ROOT = "/content/drive/MyDrive/BanglaIndoorVQA"   # the Drive project folder
REPO_SLUG = "OWNER/banglaindoorvqa-main"                 # set OWNER
MAIN = "/content/banglaindoorvqa-main"                   # the main clone
GATE_E = "/content/bivqa_gateE"                          # detached checkout of the tag gate-E
SCRIPT = MAIN + "/scripts/scoring_audit.py"

## Cell 0: clone, pull, gate-E checkout, input check
Mounts Drive, clones `banglaindoorvqa-main` with the Colab secret `GITHUB_TOKEN` (the token is
never printed or written to disk), pulls, makes the detached gate-E checkout, then runs the
read-only check.

**Correct output:** the Git steps end with `SETUP: PASS`, then lines C01 to C30, each `PASS`
or `INFO`, then `CHECK: ALL PASS (30 lines)`. C05 should read `tests/: same as gate-E`.
C27 lists the kept harness fields. A line `NOTE: the pilot scoring kept fields that these
caches lack` is not a fail, but send it in the chat. Any `FAIL`: stop and send the whole
printout.

In [ ]:
# Cell 0: setup and the input check
import os, subprocess, sys
from google.colab import drive, userdata

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"      # no __pycache__ in the gate-E checkout (D252)
sys.dont_write_bytecode = True
drive.mount("/content/drive")

if "OWNER" in REPO_SLUG:
    raise SystemExit("FAIL: set REPO_SLUG in Cell P")
if not REPO_SLUG.endswith("/banglaindoorvqa-main"):
    raise SystemExit("FAIL: REPO_SLUG must name banglaindoorvqa-main, not the dataset repo")
try:
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception as exc:
    raise SystemExit("FAIL: the Colab secret GITHUB_TOKEN is missing or not allowed for this "
                     "notebook (%s)" % type(exc).__name__)
if not TOKEN:
    raise SystemExit("FAIL: the Colab secret GITHUB_TOKEN is empty")
PLAIN = "https://github.com/%s.git" % REPO_SLUG
AUTHED = "https://x-access-token:%s@github.com/%s.git" % (TOKEN, REPO_SLUG)


def sh(*args):
    p = subprocess.run(list(args), capture_output=True, text=True)
    shown = " ".join(args).replace(TOKEN, "***")
    out = (p.stdout + p.stderr).replace(TOKEN, "***").strip()
    print("%s  %s%s" % ("PASS" if p.returncode == 0 else "FAIL", shown,
                        ("\n" + out) if out else ""))
    if p.returncode:
        raise SystemExit("SETUP: FAIL (exit %d)" % p.returncode)
    return out


if not os.path.isdir(MAIN + "/.git"):
    sh("git", "clone", "-q", AUTHED, MAIN)
sh("git", "-C", MAIN, "remote", "set-url", "origin", PLAIN)       # no token in .git/config
sh("git", "-C", MAIN, "checkout", "-q", "main")
sh("git", "-C", MAIN, "fetch", "-q", AUTHED,
   "+refs/heads/main:refs/remotes/origin/main", "+refs/tags/*:refs/tags/*")
sh("git", "-C", MAIN, "merge", "-q", "--ff-only", "origin/main")   # the pull
sh("git", "-C", MAIN, "worktree", "prune")
if not os.path.isdir(GATE_E):
    sh("git", "-C", MAIN, "worktree", "add", "-q", "--detach", GATE_E, "gate-E")
print("main HEAD", sh("git", "-C", MAIN, "rev-parse", "--short=12", "HEAD").splitlines()[-1])
print("gate-E checkout HEAD", sh("git", "-C", GATE_E, "rev-parse", "--short=12", "HEAD").splitlines()[-1])
print("SETUP: PASS")


def run(*args):
    cmd = [sys.executable, SCRIPT, "--root", DRIVE_ROOT, "--gate-e", GATE_E, "--main", MAIN] + list(args)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env=dict(os.environ, PYTHONDONTWRITEBYTECODE="1"))
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    print("exit code %d" % rc)
    return rc


run("check")

## Cell 1: selftest on made-up answers
Builds a small made-up project in a temporary folder and runs the whole chain with the real
gate-E scorer. Reads no project data.

**Correct output:** `SELFTEST: 27 of 27 checks PASS`.

In [ ]:
# Cell 1: selftest
run("selftest")

## Cell 2: score the six caches
One gate-E `asset9_scorer.py score` command per cache, run from the gate-E checkout (D270,
D271). Each is checked against its cache straight after. A scored file that exists is skipped;
`--overwrite` replaces it, but never once an HT-4 sheet exists.

**Correct output:** for each model the scorer's own printout, then a `PASS` line:

| model | records | scored | excluded |
|---|---|---|---|
| qwen | 15,947 | 15,790 | 157 |
| gemma | 11,600 | 11,485 | 115 |
| gemini | 11,600 | 11,485 | 115 |
| qwen_ft_s42, s43, s44 | 4,347 each | 4,305 each | 42 each |

(Excluded: 9 per P-CORE-CLOSED condition, 9 per P-NAME and P-CLUST condition, 8 per P-FCE
condition, 14 per R-TEST condition.)

qwen also prints `free check: 738 of 738 copies score exactly as their EXP-L source (6 of them
excluded items), 0 differ`. The cell ends with `SCORE: qwen PASS, gemma PASS, ...` for all six.
`Skipped` must not appear in any scorer printout.

In [ ]:
# Cell 2: score (one model at a time is fine: run("score", "--model", "gemma"))
run("score")

## Cell 3: verify
Reads the six scored files again and checks them against their caches and manifests; repeats
the free check of the 738 copies (addendum L10). Read-only.

**Correct output:** six `PASS` lines with the counts of Cell 2, the free-check line, then
`All six files: 52188 records, 51675 scored, 513 excluded` and `VERIFY: ALL PASS`.

In [ ]:
# Cell 3: verify
run("verify")

## Cell 4: D192, D194 and open-issue counts (D272)
Counts only. The answers behind the counts go to `scored/main/reading/TEXT_*.csv` for HT-4
and HT-5; do not paste those files in the chat.

**Correct output:** a D192 table with seven findings (d192_5 and d192_6 split by gold), a
D194 table per model, the forced-choice open issues per model and condition, and `COUNTS: DONE`.
These are results; there is no expected value.

In [ ]:
# Cell 4: counts
run("counts")

## Cell 5: D159 against main (D273)
The 97 D5 pairs (L-BB 58, N-BB 39): the pilot against main (the D159 reading), and the D5
repeat and the Batch test against main beside it (information).

**Correct output:** three blocks, each `pairs 97 (L-BB 58, N-BB 39)`, and `D159: DONE`.

In [ ]:
# Cell 5: D159
run("d159")

## Cell 6: the 10-item check
Ten seeded-random scored answers: 4 open with the image, 2 without, 4 forced choice (one per
forced-choice condition), the make-up of D166.

**Correct output:** ten lines S01 to S10 (model, condition, qa_id) and the path of
`TEXT_sample10.csv`. Send that file in the chat.

In [ ]:
# Cell 6: sample10
run("sample10")

## Cell 7: status
What exists so far. Read-only.

In [ ]:
# Cell 7: status
run("status")